<a href="https://colab.research.google.com/github/Apekshagaonkar/ai-systems-notebooks/blob/main/week7_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 — AI Agents
### From a model that can only *talk* to one that can *do*

Weeks 5 and 6: the LLM reads what we send it and writes an answer. One round trip.
This week, the model gets **tools** and a **loop**. It decides what to do, our code does it,
the model looks at the result, and decides again until the goal is done.

**No API key needed.** A free *pretend* model plays the agent. With an OpenAI key, the same loop runs on the real model.

## Setup — run this first
Creates today's (made-up) weather data and picks the model: the **real** OpenAI model if you've added a key, otherwise the free **pretend** agent.

*Optional:* to use a real key in Colab, click 🔑 (Secrets) on the left, add `OPENAI_API_KEY`, and turn on notebook access.

In [ ]:
!pip -q install openai

import os, re, json, inspect, sqlite3
from types import SimpleNamespace
import pandas as pd

MODEL = "gpt-4o"   # change it if your provider has retired this model

# ---- today's weather, per city (made up, so everyone gets the same answers) ----
CITY_WEATHER = {
    "Seattle":  {"temp_c": 12, "rain_mm": 3.2, "wind_kmh": 18, "sky": "light rain"},
    "Boston":   {"temp_c": 19, "rain_mm": 0.0, "wind_kmh": 14, "sky": "sunny"},
    "San Jose": {"temp_c": 24, "rain_mm": 0.0, "wind_kmh": 10, "sky": "clear"},
    "Phoenix":  {"temp_c": 38, "rain_mm": 0.0, "wind_kmh": 12, "sky": "very hot"},
}

def is_good_day(temp_c, rain_mm, wind_kmh):
    """Our Week 3 rule: mild, dry and not too windy."""
    return 4 <= temp_c <= 28 and rain_mm < 1.0 and wind_kmh < 24

# ---- the pretend model (it plays the agent with a simple script) ----
class MockLLM:
    """Same shape as the real OpenAI client, including tool calls.
    A real model DECIDES its next step; this one follows a short script so the notebook runs for free."""
    def __init__(self):
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
        self._n = 0

    def _call(self, name, args):
        self._n += 1
        fn = SimpleNamespace(name=name, arguments=json.dumps(args))
        return SimpleNamespace(id=f"call_{self._n}", type="function", function=fn)

    def _reply(self, content=None, tool_calls=None):
        msg = SimpleNamespace(role="assistant", content=content, tool_calls=tool_calls)
        return SimpleNamespace(choices=[SimpleNamespace(message=msg,
                               finish_reason="tool_calls" if tool_calls else "stop")])

    def _create(self, model, messages, tools=None, **kwargs):
        goal = next(m["content"] for m in messages if m["role"] == "user")
        names = [t["function"]["name"] for t in (tools or [])]
        call_names = {c["id"]: c["function"]["name"]
                      for m in messages if m["role"] == "assistant" for c in (m.get("tool_calls") or [])}
        results = [(call_names[m["tool_call_id"]], m["content"]) for m in messages if m["role"] == "tool"]

        if not names:                                   # a plain call: it can only talk
            return self._reply("I can't check live weather. I don't have access to current data, "
                               "so I can only tell you what weather is usually like there.")

        if "get_weather" in names:
            asked = [json.loads(c["function"]["arguments"])["city"]
                     for m in messages if m["role"] == "assistant" for c in (m.get("tool_calls") or [])]
            todo = [c for c in CITY_WEATHER if c.lower() in goal.lower() and c not in asked]
            if todo:                                    # ACT: ask for every city it still needs
                return self._reply(tool_calls=[self._call("get_weather", {"city": c}) for c in todo])
            lines, good = [], []
            for _, content in results:
                w = json.loads(content)
                ok = is_good_day(w["temp_c"], w["rain_mm"], w["wind_kmh"])
                lines.append(f"{w['city']}: {w['temp_c']}°C, {w['sky']} → {'good for a walk' if ok else 'skip the walk'}")
                if ok: good.append(w["city"])
            if len(lines) > 1:
                lines.append(f"Best choice: {', '.join(good)}." if good else "Not a great day for a walk anywhere.")
            return self._reply("\n".join(lines))

        if "extract_data" in names:
            if results and results[-1][1].startswith("ERROR"):
                last_name, err = results[-1]
                if "skip_invalid" in err:               # the error told it how to recover
                    return self._reply(tool_calls=[self._call(last_name, {"skip_invalid": True})])
                if "try again" in err.lower():          # it keeps retrying (no sense of when to stop)
                    return self._reply(tool_calls=[self._call(last_name, {})])
                return self._reply(f"I could not finish the pipeline. {last_name} failed with: "
                                   f"{err[7:]} I'm stopping here so a person can check the source.")
            done = [n for n, c in results if not c.startswith("ERROR")]
            for step in ["extract_data", "transform_data", "load_to_db"]:
                if step not in done:
                    return self._reply(tool_calls=[self._call(step, {})])
            return self._reply("Done. " + " ".join(c for _, c in results if not c.startswith("ERROR")))

        if "write_python" in names:
            return self._reply(tool_calls=[self._call("write_python", {"code": (
                "by_station = {}\n"
                "for r in readings:\n"
                "    by_station.setdefault(r['station'], []).append(r['temp_c'])\n"
                "averages = {s: round(sum(t) / len(t), 1) for s, t in by_station.items()}\n"
                "result = max(averages, key=averages.get), averages")})])

        return self._reply("(pretend model) I only know the demos in this notebook.")

key = os.environ.get("OPENAI_API_KEY")
try:
    from google.colab import userdata
    key = key or userdata.get("OPENAI_API_KEY")
except Exception:
    pass

if key:
    from openai import OpenAI
    client = OpenAI(api_key=key)
    REAL_API = True
else:
    client = MockLLM()
    REAL_API = False

print("Cities with weather today:", list(CITY_WEATHER))
print("Agent brain:", f"REAL {MODEL}" if REAL_API else "pretend model (MockLLM), free, no key needed")

---
## Chapter 0

A plain LLM call, exactly like Week 5: one question in, one answer out.

In [1]:
"question = "Should I go for a walk in Seattle today?"
reply = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}])
print("Q:", question)
print("A:", reply.choices[0].message.content)

SyntaxError: unterminated string literal (detected at line 1) (2856795561.py, line 1)

---
## Chapter 1 — Tools: how a model "uses" a function

A **tool** is just a Python function we write. The model never runs it.
It *asks* us to run it, and our code decides whether to.

In [ ]:
def get_weather(city: str) -> dict:
    """Returns today's temperature (°C), rain (mm), wind (km/h) and sky for a city."""
    if city not in CITY_WEATHER:
        raise ValueError(f"No weather data for {city!r}. Known cities: {list(CITY_WEATHER)}")
    return {"city": city, **CITY_WEATHER[city]}

get_weather("Boston")

### What the model actually sees

The model can't read our Python code. It only sees a **description**: the name, the docstring, and the arguments.
Let's build that description automatically from the function itself:

In [ ]:
PY_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean"}

def describe_tool(func):
    """Turn a Python function into the tool description an LLM API expects."""
    params = inspect.signature(func).parameters
    return {"type": "function", "function": {
        "name": func.__name__,
        "description": inspect.getdoc(func),
        "parameters": {
            "type": "object",
            "properties": {name: {"type": PY_TO_JSON.get(p.annotation, "string")} for name, p in params.items()},
            "required": [name for name, p in params.items() if p.default is inspect.Parameter.empty],
        },
    }}

print(json.dumps(describe_tool(get_weather), indent=2))

### The safety boundary

The model only ever sends back a **request**: *"please call `get_weather` with `city='Seattle'`."*
**Our code** decides whether to run it. That one rule is what keeps agents safe.

---
## Chapter 2 — The ReAct loop, by hand  

**ReAct** = **Re**asoning + **Act**ing:

```
        ┌──────────────────────────────────────────────┐
        ▼                                              │
  1. REASON  — look at the goal so far, pick the next step
  2. ACT     — ask for a tool call            ─► our code runs the tool
  3. OBSERVE — read the tool's result         ◄─ result goes back in the messages
  4. REPEAT  — until the model answers without calling a tool
```

Here is the whole agent. It's a `for` loop around the Week 5 Chat Completions call:

In [ ]:
SYSTEM = ("You are a helpful agent. Use the tools when you need real data. "
          "When you have what you need, answer the user directly.")

def run_agent(goal, tools, max_steps=6, system=SYSTEM):
    registry = {f.__name__: f for f in tools}               # the tools WE allow
    schemas = [describe_tool(f) for f in tools]
    messages = [{"role": "system", "content": system}, {"role": "user", "content": goal}]
    print(f"GOAL: {goal}\n")

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=schemas)
        msg = response.choices[0].message

        if not msg.tool_calls:                               # no tool needed: it's the final answer
            print(f"[step {step}] ANSWER:\n{msg.content}")
            return msg.content

        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": c.id, "type": "function",
             "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls]})

        for call in msg.tool_calls:
            name, args = call.function.name, json.loads(call.function.arguments)
            print(f"[step {step}] REASON+ACT → {name}({args})")
            if name not in registry:                         # the safety boundary
                result = f"ERROR: {name} is not an allowed tool."
            else:
                try:
                    output = registry[name](**args)               # OUR code runs the tool
                    result = output if isinstance(output, str) else json.dumps(output)
                except Exception as e:
                    result = f"ERROR: {e}"
            print(f"[step {step}] OBSERVE   ← {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    print(f"STOPPED: no answer after {max_steps} steps.")
    return None

### Hello, agent

In [ ]:
answer = run_agent("Should I go for a walk in Seattle today?", tools=[get_weather])

In [ ]:
answer = run_agent("Should I walk in Seattle, Boston or Phoenix today? Pick the best one.", tools=[get_weather])

---
## Chapter 3 — An ETL agent, then break it  

Our pipeline's real job: every day, **extract** raw weather readings, **transform** them into daily summaries, and **load** them into a database.

Instead of a fixed script, we give the agent three tools and a goal. *(Kaggle Lesson 3)*

In [ ]:
RAW_READINGS = [   # today's readings from 3 weather stations (two are broken on purpose)
    {"station": "Downtown", "hour": 6,  "temp_c": 11.5, "rain_mm": 0.0},
    {"station": "Downtown", "hour": 12, "temp_c": 19.0, "rain_mm": 0.0},
    {"station": "Downtown", "hour": 18, "temp_c": 16.5, "rain_mm": 0.2},
    {"station": "Airport",  "hour": 6,  "temp_c": None, "rain_mm": 0.0},   # sensor dropped out
    {"station": "Airport",  "hour": 12, "temp_c": 21.0, "rain_mm": 0.0},
    {"station": "Airport",  "hour": 18, "temp_c": 17.5, "rain_mm": 0.0},
    {"station": "Hillside", "hour": 6,  "temp_c": 9.0,  "rain_mm": 0.6},
    {"station": "Hillside", "hour": 12, "temp_c": 999,  "rain_mm": 0.9},   # impossible value
    {"station": "Hillside", "hour": 18, "temp_c": 13.0, "rain_mm": 0.8},
]

STAGING = {}                                    # data stays HERE, in our code
BREAK = {"source_offline": False, "db_busy": False}
db = sqlite3.connect("weather.db")

def reset_pipeline():
    STAGING.clear()
    db.execute("DROP TABLE IF EXISTS daily_weather")

def extract_data() -> str:
    """Pulls today's raw readings from the weather source into the staging area."""
    if BREAK["source_offline"]:
        raise ConnectionError("Weather source is offline (HTTP 503). Nothing to process.")
    STAGING["raw"] = [dict(r) for r in RAW_READINGS]
    stations = sorted({r["station"] for r in STAGING["raw"]})
    return f"Extracted {len(STAGING['raw'])} readings from {len(stations)} stations: {stations}."

def transform_data(skip_invalid: bool = False) -> str:
    """Cleans the raw readings and turns them into one summary per station.
    Fails if any reading is invalid, unless skip_invalid is true."""
    if "raw" not in STAGING:
        raise RuntimeError("No raw data yet. Call extract_data first.")
    bad = [i for i, r in enumerate(STAGING["raw"]) if r["temp_c"] is None or not -50 <= r["temp_c"] <= 60]
    if bad and not skip_invalid:
        raise ValueError(f"{len(bad)} readings have a missing or impossible temperature (rows {bad}). "
                         "Call transform_data again with skip_invalid=true to drop them.")
    rows = pd.DataFrame([r for i, r in enumerate(STAGING["raw"]) if i not in bad])
    summary = rows.groupby("station").agg(avg_temp_c=("temp_c", "mean"), rain_mm=("rain_mm", "sum")).round(1)
    summary["good_day"] = [is_good_day(t, r, 10) for t, r in zip(summary.avg_temp_c, summary.rain_mm)]
    STAGING["clean"] = summary.reset_index()
    return f"Cleaned {len(rows)} readings (dropped {len(bad)}) into {len(summary)} station summaries."

def load_to_db() -> str:
    """Writes the cleaned station summaries into the daily_weather database table."""
    if BREAK["db_busy"]:
        raise RuntimeError("Database is busy. Please try again.")
    if "clean" not in STAGING:
        raise RuntimeError("No clean data yet. Call transform_data first.")
    STAGING["clean"].to_sql("daily_weather", db, if_exists="replace", index=False)
    return f"Loaded {len(STAGING['clean'])} rows into daily_weather."

ETL_TOOLS = [extract_data, transform_data, load_to_db]
print("Tools ready:", [f.__name__ for f in ETL_TOOLS])

In [ ]:
reset_pipeline()
answer = run_agent("Process today's weather data.", tools=ETL_TOOLS)
print()
pd.read_sql("SELECT * FROM daily_weather", db)

### Break it on purpose, #1: the source is down

In [ ]:
reset_pipeline()
BREAK["source_offline"] = True
answer = run_agent("Process today's weather data.", tools=ETL_TOOLS)
BREAK["source_offline"] = False

### Break it on purpose, #2: the database keeps saying "try again"

In [ ]:
reset_pipeline()
BREAK["db_busy"] = True
answer = run_agent("Process today's weather data.", tools=ETL_TOOLS, max_steps=6)
BREAK["db_busy"] = False

---
## Chapter 4 — Tool-based vs code-based agents

So far the model could only use the tools we gave it. A **code-based agent** writes its own Python and runs it.
That can solve problems we never planned a tool for, which is exciting *and* dangerous.

| Tool-based | Code-based |
|---|---|
| Calls functions we wrote | Writes and runs new code |
| Safer: we control what's possible | More powerful: handles new problems |
| Easy to audit | Needs a sandbox; harder to predict |

A tiny code agent. The question: *"Which station was warmest on average?"*. We have no tool for that.

In [ ]:
def write_python(code: str) -> str:
    """Write Python code that uses the list `readings` and stores its answer in a variable called `result`."""
    return code

BLOCKED = ["import", "open(", "exec", "eval", "__", "os.", "sys."]

def run_code_safely(code, data):
    if any(word in code for word in BLOCKED):
        return "BLOCKED: the code tried something outside the sandbox."
    safe_builtins = {"len": len, "sum": sum, "max": max, "min": min, "round": round, "sorted": sorted}
    sandbox = {"__builtins__": safe_builtins, "readings": data}
    exec(code, sandbox)                       # runs ONLY with the names we put in the sandbox
    return sandbox.get("result")

valid = [r for r in RAW_READINGS if r["temp_c"] is not None and r["temp_c"] < 60]
task = [{"role": "system", "content": "Solve the task by calling write_python. Readings have keys "
                                      "station, hour, temp_c, rain_mm. No imports."},
        {"role": "user", "content": "Which station was warmest on average?"}]
call = client.chat.completions.create(model=MODEL, messages=task,
                                      tools=[describe_tool(write_python)]).choices[0].message.tool_calls[0]
model_code = json.loads(call.function.arguments)["code"]
print("The model wrote:\n" + model_code + "\n")
print("Result:", run_code_safely(model_code, valid))

In [ ]:
# What if the code tries something we never intended?
sneaky = "import os\nresult = os.listdir('/')"
print(run_code_safely(sneaky, valid))

---
## Chapter 5 — smolagents: the framework version

Our `run_agent` had to manage the loop, the tool descriptions, the tool calls and the errors. **smolagents** (by Hugging Face) does all of that. *(Kaggle Lesson 4)*

| What we wrote | smolagents |
|---|---|
| `describe_tool(func)` | `@tool` decorator |
| `run_agent(goal, tools, max_steps)` | `agent.run(goal)` (`max_steps` is a setting) |
| our tool-based loop | `ToolCallingAgent` |
| our code sandbox | `CodeAgent` (writes and runs Python) |

**Watch the docstring:** smolagents needs an `Args:` section that describes every argument.

In [ ]:
if REAL_API:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "smolagents[openai]"], check=True)
    from smolagents import CodeAgent, tool, OpenAIServerModel

    @tool
    def city_weather(city: str) -> str:
        """Returns today's temperature, rain, wind and sky for a city.

        Args:
            city: The city name, for example "Seattle".
        """
        return json.dumps(get_weather(city))

    model = OpenAIServerModel(model_id=MODEL, api_key=key)
    agent = CodeAgent(tools=[city_weather], model=model, max_steps=5)
    print(agent.run("Should I walk in Seattle, Boston or Phoenix today? Pick the best one."))
else:
    print("Skipped: smolagents needs an OPENAI_API_KEY. Try it in Kaggle Lesson 4.")

---
## Chapter 6 — When to use an agent (and when not to)

| Use an **agent** when… | Use a plain **script** when… |
|---|---|
| The task has several steps and the order isn't fixed | The steps are fixed and well understood |
| It must adapt to unexpected results | Predictability and auditing matter most |
| It must choose between many tools | Cost and speed matter |

### AI code assistants are agents too *(Lesson 5)*
Tools like GitHub Copilot and Claude Code read your project, run tests, edit files and loop until the tests pass. Same ReAct loop, with tools like *read file*, *run tests* and *edit file*.

| Good at | Not so good at |
|---|---|
| Boilerplate and repeated patterns | New architecture decisions |
| Reading and summarising existing code | Subtle runtime bugs |
| Writing tests for known logic | Knowing when to stop |

---
## Wrap-up  

1. **Agent** = an LLM that can use tools and **loops** until the goal is done.
2. **Tool** = a function we write. The **docstring** is what the model reads. The model asks; **our code runs it.**
3. **ReAct loop** = Reason → Act → Observe → repeat. It's a `for` loop around the chat call.
4. **ETL agent** = LLM as orchestrator, tools as workers. Data stays in our code; the model sees summaries.
5. **Agents adapt, but not always well.** Clear error messages help; `max_steps` stops runaway loops; predictable failures belong in plain code.
6. **Tool-based** (safe, controlled) vs **code-based** (powerful, needs a real sandbox).
7. **Agent or script?** Fixed, nightly, auditable → script. Messy, changing, many tools → agent.

### Where this fits in our pipeline
Our Week 4 `WeatherClassifier` and Week 6 search can both become **tools**. An agent could answer
*"Plan a walk for this weekend in San Jose"* by calling the forecast, the classifier and the RAG search, in whatever order it needs.

### Think about before next time
- Which of our earlier components would you turn into a tool, and what would its docstring say?
- What should an agent *never* be allowed to do without a person approving it?
- Our agent dropped 2 bad readings and moved on. When is that the right call, and when should it stop?